# Sigma Calibration Analysis

Analyses `data/calibration_results.csv` produced by `scripts/calibration_harness.py`.

1. **Inlier ratio vs error** — is inlier_ratio a useful predictor of error?
2. **Stanford plot** — sigma_m vs error_m (is sigma calibrated?)
3. **Error map** — spatial distribution of errors on the map
4. **Bucket table** — inlier_ratio → calibrated sigma (p68) lookup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/calibration_results.csv")
ok = df[df["ok"] == True].copy()
print(f"Total: {len(df)},  OK: {len(ok)},  No-fix: {len(df) - len(ok)}")
ok.describe()

## 1. Inlier ratio vs error

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(ok["inlier_ratio"], ok["error_m"], c=ok["n_inliers"],
                cmap="viridis", s=18, alpha=0.7, edgecolors="none")
ax.set_xlabel("Inlier ratio")
ax.set_ylabel("Error (m)")
ax.set_title("Inlier ratio vs localisation error")
plt.colorbar(sc, label="n_inliers")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Stanford plot (sigma vs error)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
lim = max(ok["sigma_m"].max(), ok["error_m"].max()) * 1.1
ax.plot([0, lim], [0, lim], "k--", linewidth=1, label="sigma = error")
ax.scatter(ok["sigma_m"], ok["error_m"], s=18, alpha=0.6, edgecolors="none")
ax.set_xlabel("Reported sigma (m)")
ax.set_ylabel("True error (m)")
ax.set_title("Stanford plot — sigma calibration")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Error map

In [ ]:
import cv2

map_img = cv2.imread("../data/dresden-map.png", cv2.IMREAD_UNCHANGED)
if map_img.shape[2] == 4:
    map_rgb = cv2.cvtColor(map_img, cv2.COLOR_BGRA2RGB)
else:
    map_rgb = cv2.cvtColor(map_img, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 7.5))
ax.imshow(map_rgb, extent=[0, map_img.shape[1], map_img.shape[0], 0])
sc = ax.scatter(ok["true_x"], ok["true_y"], c=ok["error_m"],
                cmap="RdYlGn_r", s=40, alpha=0.85, edgecolors="white", linewidths=0.5)
ax.set_xlabel("Map x (px)")
ax.set_ylabel("Map y (px)")
ax.set_title("Spatial error distribution on map")
plt.colorbar(sc, label="Error (m)", shrink=0.8)
ax.set_xlim(0, map_img.shape[1])
ax.set_ylim(map_img.shape[0], 0)
plt.tight_layout()
plt.show()

## 4. Inlier-ratio bucket table → calibrated sigma lookup

In [ ]:
bins = [0, 0.15, 0.25, 0.35, 0.5, 1.01]
labels = ["0–0.15", "0.15–0.25", "0.25–0.35", "0.35–0.50", "0.50+"]
ok["bucket"] = pd.cut(ok["inlier_ratio"], bins=bins, labels=labels, right=False)

table = ok.groupby("bucket", observed=False)["error_m"].agg(
    count="count",
    median_error="median",
    p68_error=lambda s: np.percentile(s, 68) if len(s) else np.nan,
    p95_error=lambda s: np.percentile(s, 95) if len(s) else np.nan,
)
table = table.round(2)
print("Inlier-ratio bucket table")
print("=" * 60)
print(table.to_string())
print()
print("Calibrated sigma lookup (p68 error per bucket):")
for label, row in table.iterrows():
    print(f"  inlier_ratio {label:>10s}  →  sigma = {row['p68_error']:.2f} m  (n={int(row['count'])})")